# CMS Hospital Data


## Import Libraries

In [ ]:
import json
import urllib.request
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

## Download data, clean and save the dataset

In [ ]:
def process_dataset(ds):
    dist = next((d for d in ds.get("distribution", []) if d.get("mediaType") == "text/csv"), None)
    if dist:
        download_url = dist.get("downloadURL")
        title = ds.get("title")
        identifier = ds.get("identifier")
        
        # Read CSV and clean column headers to snake_case
        df = pd.read_csv(download_url, low_memory=False)
        df.columns = (
            df.columns.str.lower()
            .str.replace(r"[^\w\s]", "", regex=True)
            .str.strip()
            .str.replace(r"\s+", "_", regex=True)
        )
        
        filename = f"{identifier}.csv"
        df.to_csv(filename, index=False)
        return f"Successfully processed: {title}"
    return None

## Extract the data from the source (1), filter the data (2), Use 5 workers for parallel processing (3)

In [ ]:
#1.
url = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
with urllib.request.urlopen(url) as response:
    datasets = json.loads(response.read().decode())

# 2. 
hospital_datasets = [ds for ds in datasets if "Hospitals" in ds.get("theme", [])]
print(f"Found {len(hospital_datasets)} hospital datasets. Starting parallel download...")

# 3.
with ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(process_dataset, hospital_datasets)
    for message in results:
        if message:
            print(message)